# 03 – Detailed EDA
Deep-dive: temporal patterns, weather analysis, geographic insights, correlation analysis.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from src.visualization.visualize import (
    plot_severity_distribution, plot_temporal_trends,
    plot_weather_analysis, plot_correlation_heatmap,
    plot_state_accidents, plotly_hourly_heatmap,
    plotly_severity_by_state, plotly_top_cities
)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_parquet('../data/interim/accidents_cleaned.parquet')
print(f'Shape: {df.shape}')

## Severity Distribution

In [ ]:
plot_severity_distribution(df)
plt.show()

## Temporal Patterns

In [ ]:
plot_temporal_trends(df)
plt.show()

In [ ]:
# Interactive day x hour heatmap
fig = plotly_hourly_heatmap(df)
fig.show()

## Weather Analysis

In [ ]:
plot_weather_analysis(df)
plt.show()

In [ ]:
# Severity by weather condition
if 'Weather_Condition' in df.columns:
    top_weather = df['Weather_Condition'].value_counts().head(10).index
    wdf = df[df['Weather_Condition'].isin(top_weather)]
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=wdf, x='Weather_Condition', y='Severity', ax=ax, palette='muted')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_title('Severity Distribution by Weather Condition')
    plt.tight_layout()
    plt.show()

## Geographic Analysis

In [ ]:
plot_state_accidents(df)
plt.show()

In [ ]:
# Interactive choropleth
fig = plotly_severity_by_state(df)
fig.show()

In [ ]:
# Top cities
if 'City' in df.columns:
    fig = plotly_top_cities(df, top_n=20)
    fig.show()

## Correlation Analysis

In [ ]:
num_cols = ['Severity', 'Distance(mi)', 'Temperature(F)', 'Humidity(%)',
            'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
present = [c for c in num_cols if c in df.columns]
corr = df[present].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix – Key Numeric Features')
plt.tight_layout()
plt.show()

## Road Infrastructure Analysis

In [ ]:
infra_cols = ['Junction', 'Traffic_Signal', 'Crossing', 'Railway', 'Roundabout', 'Station', 'Stop', 'Bump']
present = [c for c in infra_cols if c in df.columns]
infra_data = []
for col in present:
    count = int(df[col].sum())
    avg_sev = df[df[col] == True]['Severity'].mean()
    infra_data.append({'Feature': col, 'Count': count, 'Avg Severity': round(avg_sev, 3)})
infra_df = pd.DataFrame(infra_data).sort_values('Count', ascending=False)
print(infra_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
infra_df.plot(x='Feature', y='Count', kind='bar', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Accidents at Road Features')
axes[0].tick_params(axis='x', rotation=45)
infra_df.plot(x='Feature', y='Avg Severity', kind='bar', ax=axes[1], color='coral', legend=False)
axes[1].set_title('Avg Severity at Road Features')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Day/Night Analysis

In [ ]:
if 'Sunrise_Sunset' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    df['Sunrise_Sunset'].value_counts().plot(kind='bar', ax=axes[0], color=['gold', 'navy'])
    axes[0].set_title('Accidents: Day vs Night')
    sns.boxplot(data=df, x='Sunrise_Sunset', y='Severity', ax=axes[1], palette=['gold', 'navy'])
    axes[1].set_title('Severity: Day vs Night')
    plt.tight_layout()
    plt.show()